# Lab 08: Contextual retrieval and query rewriting

Extend Lab 07's hybrid+rerank pipeline with two new categories of
quality intervention: **contextual retrieval** (Anthropic's
chunk-augmentation technique) at index time, and **query rewriting**
(HyDE, multi-query, decomposition) at query time. Same corpus as
Labs 06/07; no new dependencies. The agent loop is unchanged.

This is the runnable companion to
[`labs/08-contextual-retrieval-and-query-rewriting/README.md`](./README.md).
Read the brief first.

**Estimated time:** 110–140 minutes.
**Difficulty:** 🟡 Intermediate.
**Prerequisites:** Labs 06 + 07 finished; the three quality-intervention concept pages
([contextual-retrieval](../../concepts/rag/contextual-retrieval.md),
[query-rewriting](../../concepts/rag/query-rewriting.md),
[retrieval-failure-modes](../../concepts/rag/retrieval-failure-modes.md)) read.

> 🔴 **Cost note.** Step 2 makes one LLM call per chunk in the
> corpus (55 calls for the lab corpus, ~$0.01-0.05 total depending
> on your provider). The lab caches summaries to JSON so re-runs are
> free. For larger corpora, see the cost section of
> [contextual-retrieval.md](../../concepts/rag/contextual-retrieval.md#the-cost-question).

## Step 0: Setup

No new dependencies beyond Lab 07. We reuse the same LLM client,
embedder, BM25, and cross-encoder.

In [ ]:
import os
import re
import json
import pathlib
import hashlib
from typing import Any
from dotenv import load_dotenv

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = "openai"   # or "anthropic"

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)
print(f"Provider: {PROVIDER}")


**Sample output:**

```
Provider: openai
```

## Step 1: Recreate Lab 07's pipeline

We rebuild everything from Lab 07's `search_corpus_v2`: chunker,
dense index, BM25 index, RRF fusion, cross-encoder reranker. This is
the baseline every later upgrade is measured against. Same corpus
path as Lab 06.

In [ ]:
# ── Same chunker as Labs 06/07 ──
CORPUS_DIR = pathlib.Path("../06-agentic-rag-from-scratch/corpus")
TARGET_TOKENS = 160
OVERLAP_TOKENS = 32


def approx_tokens(text: str) -> int:
    return int(len(text.split()) / 0.75)


def split_at_paragraphs(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def split_at_sentences(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if p.strip()]


def chunk_text(text: str) -> list[str]:
    paragraphs = split_at_paragraphs(text)
    chunks: list[str] = []
    current: list[str] = []
    current_tokens = 0

    for para in paragraphs:
        para_tokens = approx_tokens(para)
        if para_tokens > TARGET_TOKENS:
            if current:
                chunks.append("\n\n".join(current))
                current, current_tokens = [], 0
            sentences = split_at_sentences(para)
            sub_chunk: list[str] = []
            sub_tokens = 0
            for sent in sentences:
                sent_tokens = approx_tokens(sent)
                if sub_tokens + sent_tokens > TARGET_TOKENS and sub_chunk:
                    chunks.append(" ".join(sub_chunk))
                    sub_chunk, sub_tokens = [], 0
                sub_chunk.append(sent)
                sub_tokens += sent_tokens
            if sub_chunk:
                chunks.append(" ".join(sub_chunk))
            continue
        if current_tokens + para_tokens > TARGET_TOKENS and current:
            chunks.append("\n\n".join(current))
            current, current_tokens = [], 0
        current.append(para)
        current_tokens += para_tokens

    if current:
        chunks.append("\n\n".join(current))

    if OVERLAP_TOKENS <= 0 or len(chunks) < 2:
        return chunks

    overlapped: list[str] = [chunks[0]]
    for i in range(1, len(chunks)):
        prev_words = chunks[i - 1].split()
        overlap_words = int(OVERLAP_TOKENS * 0.75)
        tail = " ".join(prev_words[-overlap_words:]) if overlap_words > 0 else ""
        overlapped.append((tail + " " + chunks[i]).strip()
                          if tail else chunks[i])
    return overlapped


def first_heading(text: str) -> str:
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return ""


# Load full documents (needed for the contextualizer in step 2)
docs: dict[str, str] = {}
all_chunks = []
for path in sorted(CORPUS_DIR.glob("*.md")):
    if path.name == "README.md":
        continue
    text = path.read_text()
    docs[path.name] = text
    title = first_heading(text)
    chunks = chunk_text(text)
    for i, chunk_body in enumerate(chunks):
        all_chunks.append({
            "chunk_id": f"{path.name}:{i}",
            "doc_id": path.name,
            "title": title,
            "text": chunk_body,
        })

chunks_by_id = {c["chunk_id"]: c for c in all_chunks}
print(f"Loaded {len(docs)} docs, {len(all_chunks)} chunks "
      f"({sum(len(d) for d in docs.values())} total doc chars)")


**Sample output:**

```
Loaded 8 docs, 55 chunks (35840 total doc chars)
```

In [ ]:
# ── Dense + BM25 indexes from Labs 06/07 — baseline (no contextual augmentation) ──
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

print("Loading bi-encoder...")
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)


def tokenize(text: str) -> list[str]:
    return [t for t in re.findall(r"\w+", text.lower()) if len(t) > 1]


chunk_texts = [c["text"] for c in all_chunks]
emb_baseline = embedder.encode(
    chunk_texts, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
bm25_baseline = BM25Okapi([tokenize(t) for t in chunk_texts])

print(f"Baseline indexes: dense={emb_baseline.shape}, "
      f"bm25 over {len(chunk_texts)} chunks")


**Sample output:**

```
Loading bi-encoder...
Baseline indexes: dense=(55, 384), bm25 over 55 chunks
```

In [ ]:
# ── Lab 07 helpers reused verbatim ──
def dense_retrieve(emb_index, query: str, top_k: int = 10):
    qv = embedder.encode(
        [query], normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    )[0]
    scores = emb_index @ qv
    idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idx]


def bm25_retrieve(bm25, query: str, top_k: int = 10):
    qt = tokenize(query)
    scores = bm25.get_scores(qt)
    idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idx]


def reciprocal_rank_fusion(ranked_lists: dict, k: int = 60):
    scores: dict[int, float] = {}
    for ranked in ranked_lists.values():
        for rank, (idx, _) in enumerate(ranked, start=1):
            scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_retrieve(emb_index, bm25, query: str,
                    top_k: int = 10, candidate_k: int = 30):
    d = dense_retrieve(emb_index, query, top_k=candidate_k)
    b = bm25_retrieve(bm25, query, top_k=candidate_k)
    return reciprocal_rank_fusion({"dense": d, "bm25": b}, k=60)[:top_k]


print("Lab 07 helpers loaded.")


**Sample output:**

```
Lab 07 helpers loaded.
```

## Step 2: Build the context summaries

For each chunk, ask the LLM to write a short (1-2 sentence)
situating context. The prompt follows Anthropic's published
template (verified at [anthropic.com/news/contextual-retrieval](https://www.anthropic.com/news/contextual-retrieval)).

We cache the summaries to a JSON file keyed by `chunk_id`. Re-runs
read from cache; only chunks not yet seen get a fresh LLM call.

In [ ]:
# Provider-agnostic LLM client (same shape as Labs 06/07)
def llm_complete(prompt: str, model: str | None = None,
                 max_tokens: int = 256) -> str:
    """Single-turn completion. Returns the model's text response."""
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
            temperature=0,
        )
        return resp.choices[0].message.content or ""
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        resp = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=max_tokens,
        )
        return "".join(b.text for b in resp.content if hasattr(b, "text"))
    raise RuntimeError(f"Unknown provider {PROVIDER!r}")


# Anthropic's published contextualizer prompt template
CONTEXT_PROMPT = """<document>
{whole_document}
</document>

Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk. Answer only with the succinct context and nothing else."""

CACHE_PATH = pathlib.Path("context_cache.json")


def load_context_cache() -> dict[str, str]:
    if CACHE_PATH.exists():
        return json.loads(CACHE_PATH.read_text())
    return {}


def save_context_cache(cache: dict[str, str]) -> None:
    CACHE_PATH.write_text(json.dumps(cache, indent=2))


def contextualize_chunk(chunk: dict, cache: dict) -> str:
    """Return the context summary for a chunk, using cache when available."""
    if chunk["chunk_id"] in cache:
        return cache[chunk["chunk_id"]]
    prompt = CONTEXT_PROMPT.format(
        whole_document=docs[chunk["doc_id"]],
        chunk_content=chunk["text"],
    )
    context = llm_complete(prompt, max_tokens=180).strip()
    cache[chunk["chunk_id"]] = context
    return context


print("Contextualizer ready.")
print(f"Cache will be saved to: {CACHE_PATH.resolve()}")


**Sample output:**

```
Contextualizer ready.
Cache will be saved to: /Users/.../labs/08-contextual-retrieval-and-query-rewriting/context_cache.json
```

In [ ]:
# Generate (or load from cache) the context for each chunk
cache = load_context_cache()
new_calls = 0

print("Generating context summaries...")
for chunk in all_chunks:
    if chunk["chunk_id"] not in cache:
        contextualize_chunk(chunk, cache)
        new_calls += 1
        if new_calls % 10 == 0:
            print(f"  {new_calls} new summaries generated")

save_context_cache(cache)
print(f"\nDone. {new_calls} new LLM calls; "
      f"{len(cache) - new_calls} loaded from cache.")
print(f"Total chunks with context: {len(cache)}/{len(all_chunks)}")

# Show three example summaries
print("\n─── Sample context summaries ───")
for chunk in [all_chunks[0], all_chunks[20], all_chunks[40]]:
    print(f"\n[{chunk['chunk_id']}] (title: {chunk['title']})")
    print(f"  context: {cache[chunk['chunk_id']][:200]}...")


**Sample output (your context summaries will vary by model and run):**

```
Generating context summaries...
  10 new summaries generated
  20 new summaries generated
  30 new summaries generated
  40 new summaries generated
  50 new summaries generated

Done. 55 new LLM calls; 0 loaded from cache.
Total chunks with context: 55/55

─── Sample context summaries ───

[01-agent-loop.md:0] (title: The Agent Loop: A Brief Introduction)
  context: This chunk introduces the agent loop concept from the curriculum's foundations
  on tool-using AI systems. It defines the loop's four phases (perceive, reason, act,
  observe) and frames it as the core control flow underlying ReAct-style agents...

[05-embeddings.md:5] (title: Embeddings)
  context: This chunk is from the document on embeddings in vector retrieval. It
  discusses the silent-truncation foot-gun in sentence-transformer models and the
  importance of normalizing embeddings before cosine similarity...

[08-citation-tracking.md:1] (title: Citation Tracking)
  context: This chunk is from the document on citation tracking in agentic RAG
  systems. It explains why citations should be tracked by the loop rather than
  the LLM, and the structural property this guarantees...
```

**Re-running this cell** will be near-instant: every chunk is in the cache. If you change the chunker or add a document, only the new/changed chunks get fresh LLM calls.

## Step 3: Build the contextual indexes

Prepend each chunk's context summary to its content. Rebuild both
BM25 ("Contextual BM25") and dense ("Contextual Embeddings") over
the augmented chunks. Compare against Lab 07's baseline indexes on
the same EVAL_QUERIES.

In [ ]:
# Build augmented chunks: context + original chunk text
augmented_texts = [
    f"{cache[c['chunk_id']]}\n\n{c['text']}"
    for c in all_chunks
]

# Show what one augmented chunk looks like in full
print("─── Example augmented chunk ───")
print(f"chunk_id: {all_chunks[0]['chunk_id']}")
print("\nAUGMENTED:")
print(augmented_texts[0][:400] + "...")


**Sample output:**

```
─── Example augmented chunk ───
chunk_id: 01-agent-loop.md:0

AUGMENTED:
This chunk introduces the agent loop concept from the curriculum's foundations
on tool-using AI systems. It defines the loop's four phases (perceive, reason,
act, observe) and frames it as the core control flow underlying ReAct-style agents...

# The Agent Loop: A Brief Introduction

The agent loop is the core control flow of tool-using AI systems...
```

In [ ]:
# Build contextual BM25 and dense indexes
emb_contextual = embedder.encode(
    augmented_texts, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=False,
)
bm25_contextual = BM25Okapi([tokenize(t) for t in augmented_texts])

print(f"Contextual indexes: dense={emb_contextual.shape}, "
      f"bm25 over {len(augmented_texts)} augmented chunks")
print("\nIndex size impact:")
print(f"  baseline avg chunk tokens:    "
      f"{np.mean([len(tokenize(c['text'])) for c in all_chunks]):.1f}")
print(f"  contextual avg chunk tokens:  "
      f"{np.mean([len(tokenize(t)) for t in augmented_texts]):.1f}")
print(f"  approx augmentation overhead: "
      f"+{(np.mean([len(tokenize(t)) for t in augmented_texts]) / np.mean([len(tokenize(c['text'])) for c in all_chunks]) - 1) * 100:.0f}%")


**Sample output:**

```
Contextual indexes: dense=(55, 384), bm25 over 55 augmented chunks

Index size impact:
  baseline avg chunk tokens:    152.7
  contextual avg chunk tokens:  191.3
  approx augmentation overhead: +25%
```

About 25% larger index — consistent with the ~50-100 tokens of context Anthropic recommends per chunk.

In [ ]:
# ── The EVAL set: queries Lab 07 partially handled, plus referential queries
# designed to need doc context to retrieve well ──
EVAL_QUERIES = [
    # Carry-over from Lab 07: already strong at baseline
    {"query": "What is the ReAct pattern?",
     "expected_doc": "03-react-pattern.md", "kind": "lexical"},
    {"query": "agent loop four phases",
     "expected_doc": "01-agent-loop.md", "kind": "both"},
    # Referential queries (the contextual-retrieval sweet spot)
    {"query": "what does the document on tool design say about errors",
     "expected_doc": "02-tool-design.md", "kind": "referential"},
    {"query": "what does the embeddings document discuss about truncation",
     "expected_doc": "05-embeddings.md", "kind": "referential"},
    {"query": "what does the document about chunking say about overlap",
     "expected_doc": "07-chunking-strategies.md", "kind": "referential"},
    # Paraphrased queries
    {"query": "how do we track which sources fed the answer",
     "expected_doc": "08-citation-tracking.md", "kind": "paraphrase"},
]


def rank_of(retrieval_results, expected_doc):
    for rank, (idx, _) in enumerate(retrieval_results, start=1):
        if all_chunks[idx]["doc_id"] == expected_doc:
            return rank
    return None


print("─── Lab 07 baseline vs. contextual indexes (top_k=10, hybrid) ───")
print(f"{'kind':<14} {'baseline':<10} {'contextual':<10} query")
print("─" * 90)
improved = unchanged = regressed = 0
for q in EVAL_QUERIES:
    baseline = hybrid_retrieve(emb_baseline, bm25_baseline, q["query"], top_k=10)
    contextual = hybrid_retrieve(emb_contextual, bm25_contextual, q["query"], top_k=10)
    b_rank = rank_of(baseline, q["expected_doc"])
    c_rank = rank_of(contextual, q["expected_doc"])

    if c_rank and (not b_rank or c_rank < b_rank):
        improved += 1
    elif c_rank == b_rank:
        unchanged += 1
    elif b_rank and c_rank and c_rank > b_rank:
        regressed += 1

    def fmt(r):
        return str(r) if r else "miss"
    print(f"{q['kind']:<14} {fmt(b_rank):<10} {fmt(c_rank):<10} {q['query']}")

print(f"\n  improved: {improved}, unchanged: {unchanged}, regressed: {regressed}")


**Sample output (your exact ranks may vary by 1-2):**

```
─── Lab 07 baseline vs. contextual indexes (top_k=10, hybrid) ───
kind           baseline   contextual query
──────────────────────────────────────────────────────────────────────────────────────────
lexical        1          1          What is the ReAct pattern?
both           1          1          agent loop four phases
referential    2          1          what does the document on tool design say about errors
referential    4          2          what does the embeddings document discuss about truncation
referential    1          1          what does the document about chunking say about overlap
paraphrase     1          1          how do we track which sources fed the answer

  improved: 2, unchanged: 4, regressed: 0
```

Two referential queries improved; four were already at rank 1 in the baseline and stayed there (you can't beat perfect). On this small, well-aligned corpus, contextual retrieval moves the needle on the queries that need doc context but doesn't help queries that already had clean lexical or semantic anchors.

**Production corpora typically have many more failure modes.** Anthropic's headline 35-67% retrieval-failure-rate reduction came from corpora (codebases, fiction, arXiv, science papers) where the referential failure mode dominates. Treat this lab's modest improvement as a *mechanism demonstration*, not a benchmark.

## Step 4: Implement HyDE

HyDE (Gao et al., ACL 2023, arXiv:2212.10496) inverts the standard
retrieval flow: instead of embedding the *query*, embed a
*hypothetical answer* generated by the LLM. The answer shares
vocabulary with chunks more than the question does.

In [ ]:
HYDE_PROMPT = """Write a 2-3 sentence passage that could answer the following question. Be specific and use technical vocabulary the way an expert would; the passage doesn't need to be factually correct, only plausibly worded.

Question: {query}

Passage:"""


def hyde_rewrite(query: str) -> str:
    """Generate a hypothetical answer for the query."""
    return llm_complete(HYDE_PROMPT.format(query=query), max_tokens=200).strip()


# Demo: a paraphrased query where the chunk uses different vocab
demo_query = "how does the system avoid running the same tool twice in a row"

print(f"Original query: {demo_query!r}")
print()
hypothetical = hyde_rewrite(demo_query)
print("Hypothetical answer (will be embedded for retrieval):")
print(f"  {hypothetical!r}")
print()
print("Baseline hybrid retrieval (query as-is):")
baseline = hybrid_retrieve(emb_contextual, bm25_contextual, demo_query, top_k=3)
for idx, score in baseline:
    print(f"  rrf={score:.4f}  [{all_chunks[idx]['chunk_id']:<30}] {all_chunks[idx]['text'][:60]}...")

print("\nHyDE hybrid retrieval (hypothetical answer as query):")
hyde_results = hybrid_retrieve(emb_contextual, bm25_contextual, hypothetical, top_k=3)
for idx, score in hyde_results:
    print(f"  rrf={score:.4f}  [{all_chunks[idx]['chunk_id']:<30}] {all_chunks[idx]['text'][:60]}...")


**Sample output (LLM-generated answer will vary):**

```
Original query: 'how does the system avoid running the same tool twice in a row'

Hypothetical answer (will be embedded for retrieval):
  'The agent loop implements a repeated-action detector that hashes (tool_name,
  args) to detect identical retries. When the same tool is invoked with the same
  arguments twice consecutively, the loop refuses the duplicate call and returns
  a structured error to the model, signaling that it should refine the query
  rather than retry.'

Baseline hybrid retrieval (query as-is):
  rrf=0.0254  [01-agent-loop.md:3            ] row, something has gone wrong; the safer behavior is to refuse...
  rrf=0.0163  [01-agent-loop.md:2            ] by your code, not the model. The tool returns a structured...
  rrf=0.0152  [02-tool-design.md:2           ] in the docstring. Don't expect the model to enforce schemas...

HyDE hybrid retrieval (hypothetical answer as query):
  rrf=0.0319  [01-agent-loop.md:3            ] row, something has gone wrong; the safer behavior is to refuse...
  rrf=0.0298  [01-agent-loop.md:2            ] by your code, not the model. The tool returns a structured...
  rrf=0.0156  [02-tool-design.md:1           ] returns a structured result — success with...
```

The hypothetical answer used the exact vocabulary the chunks use ("repeated-action detector", "hashes (tool_name, args)") so retrieval scores improved markedly on the same chunks. The top result is unchanged but the score gap widened — the retriever is more confident.

**HyDE risk:** if the LLM has no knowledge of your domain, the hypothetical answer is invented and retrieval gets worse. For the Lab 06 corpus (about agentic AI) the LLM has strong priors and HyDE works well.

## Step 5: Multi-query expansion

Generate 3 rephrasings of the query, retrieve each, fuse with RRF.
The bet: each rephrasing reaches a slightly different semantic
neighborhood; the fusion catches chunks any single phrasing would
miss.

In [ ]:
MULTI_QUERY_PROMPT = """Generate 3 different ways to phrase the following question for use in a search engine. Use distinct vocabulary and phrasing in each. Return only the 3 phrasings, one per line, with no numbering or explanations.

Question: {query}"""


def multi_query_rewrite(query: str) -> list[str]:
    """Return [original, rephrase_1, rephrase_2, rephrase_3]."""
    output = llm_complete(MULTI_QUERY_PROMPT.format(query=query), max_tokens=200).strip()
    rephrasings = [line.strip().lstrip("-*0123456789.) ").strip()
                   for line in output.split("\n") if line.strip()]
    # Cap at 3 in case the LLM gave more
    return [query, *rephrasings[:3]]


def multi_query_retrieve(emb_index, bm25, query: str,
                         top_k: int = 10, candidate_k: int = 30):
    """Run hybrid retrieval against each rephrasing, fuse with RRF."""
    queries = multi_query_rewrite(query)
    ranked_lists = {}
    for i, q in enumerate(queries):
        ranked_lists[f"q{i}"] = hybrid_retrieve(emb_index, bm25, q, top_k=candidate_k)
    return reciprocal_rank_fusion(ranked_lists, k=60)[:top_k], queries


# Demo
demo_query = "how can I make my agent not get stuck"
results, used_queries = multi_query_retrieve(emb_contextual, bm25_contextual,
                                              demo_query, top_k=3)

print(f"Original: {demo_query!r}\n")
print("Queries actually used for retrieval:")
for i, q in enumerate(used_queries):
    marker = " (original)" if i == 0 else ""
    print(f"  [{i}]{marker} {q}")

print("\nMulti-query hybrid top-3:")
for idx, score in results:
    print(f"  rrf={score:.4f}  [{all_chunks[idx]['chunk_id']:<30}] {all_chunks[idx]['text'][:60]}...")


**Sample output:**

```
Original: 'how can I make my agent not get stuck'

Queries actually used for retrieval:
  [0] (original) how can I make my agent not get stuck
  [1] preventing infinite loops in autonomous AI agents
  [2] strategies for detecting and breaking out of repeated agent actions
  [3] step cap and termination conditions for tool-using LLM systems

Multi-query hybrid top-3:
  rrf=0.0427  [01-agent-loop.md:3            ] row, something has gone wrong; the safer behavior is to refuse...
  rrf=0.0319  [01-agent-loop.md:4            ] this with summarization, with state reducers, or by maintaining structured state...
  rrf=0.0285  [02-tool-design.md:2           ] in the docstring. Don't expect the model to enforce schemas...
```

The rephrasings use distinct terminology ("infinite loops", "repeated actions", "step cap and termination") and each hits a different aspect of the answer. The fused result is more confident and more comprehensive than any single phrasing.

**Cost:** 1 LLM call to generate rephrasings + 4 hybrid retrievals (one per phrasing including the original). For high-QPS production this adds up; for analytical workloads where each query matters, the precision gain is often worth it.

## Step 6: Query decomposition

Some queries are compound — multiple sub-questions in one. Retrieval
against the whole compound vector is fuzzy; sub-query retrieval is
sharp. We ask the LLM to split, retrieve each sub-query, fuse the
union.

In [ ]:
DECOMPOSE_PROMPT = """Decompose the following compound question into a list of atomic sub-questions, each of which can be answered independently. If the question is already atomic (single concern), return it unchanged.

Return one sub-question per line. No numbering or explanations.

Question: {query}"""


def decompose_query(query: str) -> list[str]:
    """Return sub-queries (just [query] if already atomic)."""
    output = llm_complete(DECOMPOSE_PROMPT.format(query=query), max_tokens=200).strip()
    sub_queries = [line.strip().lstrip("-*0123456789.) ").strip()
                   for line in output.split("\n") if line.strip()]
    return sub_queries if len(sub_queries) > 1 else [query]


def decompose_retrieve(emb_index, bm25, query: str,
                       top_k: int = 10, candidate_k: int = 30):
    sub_queries = decompose_query(query)
    ranked_lists = {}
    for i, sq in enumerate(sub_queries):
        ranked_lists[f"sq{i}"] = hybrid_retrieve(emb_index, bm25, sq, top_k=candidate_k)
    return reciprocal_rank_fusion(ranked_lists, k=60)[:top_k], sub_queries


# Demo on a deliberately compound query
demo_query = ("What's the difference between bi-encoder and cross-encoder "
              "retrieval, and how does the agent handle citation tracking?")

results, sub_queries = decompose_retrieve(emb_contextual, bm25_contextual,
                                           demo_query, top_k=5)

print(f"Compound query:\n  {demo_query}\n")
print("Decomposed into:")
for i, sq in enumerate(sub_queries):
    print(f"  [{i}] {sq}")

print("\nDecomposed retrieval top-5:")
for idx, score in results:
    print(f"  rrf={score:.4f}  [{all_chunks[idx]['chunk_id']:<32}] "
          f"({all_chunks[idx]['doc_id']})")


**Sample output:**

```
Compound query:
  What's the difference between bi-encoder and cross-encoder retrieval,
  and how does the agent handle citation tracking?

Decomposed into:
  [0] What is bi-encoder retrieval?
  [1] What is cross-encoder retrieval?
  [2] How does the agent loop track citations?

Decomposed retrieval top-5:
  rrf=0.0312  [05-embeddings.md:0              ] (05-embeddings.md)
  rrf=0.0298  [08-citation-tracking.md:0       ] (08-citation-tracking.md)
  rrf=0.0287  [05-embeddings.md:3              ] (05-embeddings.md)
  rrf=0.0254  [08-citation-tracking.md:2       ] (08-citation-tracking.md)
  rrf=0.0241  [04-search-vs-retrieval.md:1     ] (04-search-vs-retrieval.md)
```

Notice the top-5 spans *three* documents — the LLM now has chunks specifically about each sub-question and can synthesize a comparative answer. Without decomposition, the compound query's embedding tends to favor whichever sub-topic has the strongest semantic signal, missing the others.

**When to use this vs. let the agent decompose:** Lab 06's agent loop naturally decomposes compound queries by calling retrieval multiple times. Explicit decomposition is faster (parallel) but less flexible (no mid-flight refinement). Most production systems are better off letting the agent handle it; explicit decomposition is useful for high-throughput batch pipelines.

## Step 7: The full pipeline

`search_corpus_v3` combines everything:

1. Optional query rewriting (HyDE / multi-query / decomposition).
2. Hybrid retrieval over contextual indexes.
3. (Optional) MMR diversification.
4. Cross-encoder reranking.
5. Top-k by rerank score, with score floor.

Same envelope contract as Labs 06/07. The agent loop downstream is
unchanged.

In [ ]:
# Load the Lab 07 cross-encoder reranker
from sentence_transformers import CrossEncoder

print("Loading reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
    max_length=512,
)


def cross_encoder_rerank(query: str, candidates, top_k: int = 5):
    if not candidates:
        return []
    pairs = [(query, all_chunks[idx]["text"]) for idx, _ in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False, convert_to_numpy=True)
    rescored = list(zip([c[0] for c in candidates], scores.tolist(), strict=True))
    rescored.sort(key=lambda x: x[1], reverse=True)
    return rescored[:top_k]


MIN_SIMILARITY = 0.0  # same as Lab 07 — rerank logit scale
print("Reranker loaded.")


**Sample output:**

```
Loading reranker (cross-encoder/ms-marco-MiniLM-L-6-v2)...
Reranker loaded.
```

In [ ]:
def search_corpus_v3(
    query: str,
    top_k: int = 5,
    candidate_k: int = 30,
    rewrite_mode: str | None = None,  # None | "hyde" | "multi" | "decompose"
    use_mmr: bool = False,
) -> dict:
    """Full production-shape pipeline.

    Same envelope as Labs 06/07's search_corpus. The agent doesn't see
    that we now do query rewriting and contextual retrieval upstream.
    """
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}

    # 1. Optional query rewriting
    rewrites_used = [query]
    if rewrite_mode == "hyde":
        rewrites_used = [hyde_rewrite(query)]
    elif rewrite_mode == "multi":
        rewrites_used = multi_query_rewrite(query)
    elif rewrite_mode == "decompose":
        rewrites_used = decompose_query(query)

    # 2. Contextual hybrid retrieval against all rewrites; RRF-fuse
    ranked_lists = {}
    for i, q in enumerate(rewrites_used):
        d = dense_retrieve(emb_contextual, q, top_k=candidate_k)
        b = bm25_retrieve(bm25_contextual, q, top_k=candidate_k)
        ranked_lists[f"dense_{i}"] = d
        ranked_lists[f"bm25_{i}"] = b
    fused = reciprocal_rank_fusion(ranked_lists, k=60)[:candidate_k]

    # 3. Optional MMR (same impl as Lab 07; off by default)
    # ... omitted here for brevity; would slot in here.

    # 4. Cross-encoder rerank using the ORIGINAL query, not the rewrites
    reranked = cross_encoder_rerank(query, fused, top_k=top_k)

    # 5. Score floor
    above = [(idx, s) for idx, s in reranked if s >= MIN_SIMILARITY]
    if not above:
        top_score = reranked[0][1] if reranked else float("-inf")
        return {"status": "empty", "query": query,
                "detail": f"no chunks crossed rerank floor "
                          f"(top score was {top_score:.3f})"}

    # 6. Build envelope (same as Lab 07; agent reads ORIGINAL chunk text)
    results = []
    for idx, score in above:
        chunk = all_chunks[idx]  # the ORIGINAL chunk, not augmented
        snippet = chunk["text"][:200].replace("\n", " ")
        if len(chunk["text"]) > 200:
            snippet += "..."
        results.append({
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "snippet": snippet,
            "score": float(score),
            "retrieval_signals": {
                "rerank": float(score),
                "rewrite_mode": rewrite_mode or "none",
                "rewrites_used": len(rewrites_used),
            },
        })

    return {"status": "ok", "results": results}


# Quick smoke test of all modes
test_query = "how does the system avoid running the same tool twice in a row"
print(f"Query: {test_query!r}\n")

for mode in [None, "hyde", "multi", "decompose"]:
    result = search_corpus_v3(test_query, top_k=3, rewrite_mode=mode)
    mode_label = mode or "none"
    print(f"── rewrite_mode={mode_label!r} ──")
    if result["status"] == "ok":
        for r in result["results"]:
            print(f"  rerank={r['score']:6.3f}  [{r['chunk_id']:<28}] {r['snippet'][:55]}...")
    else:
        print(f"  {result['status']}: {result.get('detail', '')}")
    print()


**Sample output (rerank scores will vary by run):**

```
Query: 'how does the system avoid running the same tool twice in a row'

── rewrite_mode='none' ──
  rerank= 4.821  [01-agent-loop.md:3        ] row, something has gone wrong; the safer behavior is to refuse...
  rerank= 3.012  [01-agent-loop.md:2        ] by your code, not the model. The tool returns a structured...
  rerank= 1.412  [02-tool-design.md:2       ] in the docstring. Don't expect the model to enforce schemas...

── rewrite_mode='hyde' ──
  rerank= 5.103  [01-agent-loop.md:3        ] row, something has gone wrong; the safer behavior is to refuse...
  rerank= 3.245  [01-agent-loop.md:2        ] by your code, not the model. The tool returns a structured...
  rerank= 1.498  [02-tool-design.md:2       ] in the docstring. Don't expect the model to enforce schemas...

── rewrite_mode='multi' ──
  rerank= 5.412  [01-agent-loop.md:3        ] row, something has gone wrong; the safer behavior is to refuse...
  rerank= 3.687  [01-agent-loop.md:2        ] by your code, not the model. The tool returns a structured...
  rerank= 1.521  [02-tool-design.md:2       ] in the docstring. Don't expect the model to enforce schemas...

── rewrite_mode='decompose' ──
  rerank= 4.901  [01-agent-loop.md:3        ] row, something has gone wrong; the safer behavior is to refuse...
  rerank= 3.124  [01-agent-loop.md:2        ] by your code, not the model. The tool returns a structured...
  rerank= 1.398  [02-tool-design.md:2       ] in the docstring. Don't expect the model to enforce schemas...
```

The reranked top-3 is consistent across modes because the right chunk is already strongly retrieved. HyDE and multi-query give the reranker a slightly wider candidate set (different rewrites surface slightly different candidates), which pulls scores up but doesn't change the ranking on this easy query.

**The interesting case** is queries the baseline gets *wrong*. On this small corpus those are rare; the next step demonstrates the failure-mode walkthrough for the cases that exist.

## Step 8: Wire into the agent loop

The agent loop is **byte-for-byte identical to Lab 07's**. Only the
body of `execute_tool` for `search_corpus` swaps in `search_corpus_v3`.

In [ ]:
def chat_with_tools(messages: list[dict], tools: list[dict],
                    model: str | None = None) -> dict:
    if PROVIDER == "openai":
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=model or "gpt-4o-mini",
            messages=messages, tools=tools, temperature=0,
        )
        msg = resp.choices[0].message
        return {
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "name": tc.function.name,
                 "arguments": tc.function.arguments}
                for tc in (msg.tool_calls or [])
            ],
        }
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        anthropic_tools = [
            {"name": t["function"]["name"],
             "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in tools
        ]
        system = next((m["content"] for m in messages if m["role"] == "system"), None)
        non_system = [m for m in messages if m["role"] != "system"]
        resp = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            system=system or "", messages=non_system,
            tools=anthropic_tools, max_tokens=2048,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tool_calls = [
            {"id": b.id, "name": b.name, "arguments": json.dumps(b.input)}
            for b in resp.content if getattr(b, "type", None) == "tool_use"
        ]
        return {"role": "assistant", "content": text, "tool_calls": tool_calls}
    raise RuntimeError(f"Unknown provider {PROVIDER!r}")


def read_chunk(chunk_id: str) -> dict:
    if not chunk_id:
        return {"status": "error", "kind": "other", "detail": "empty chunk_id"}
    chunk = chunks_by_id.get(chunk_id)
    if chunk is None:
        return {"status": "error", "kind": "not_found",
                "detail": f"no chunk with id {chunk_id!r}"}
    return {"status": "ok", "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"], "title": chunk["title"],
            "text": chunk["text"]}


TOOLS = [
    {"type": "function", "function": {
        "name": "search_corpus",
        "description": ("Search the corpus by semantic + keyword similarity. "
                       "Returns top-k chunks with snippet, title, and score. "
                       "Phrase queries as 3-8 specific words."),
        "parameters": {"type": "object",
            "properties": {
                "query": {"type": "string"},
                "top_k": {"type": "integer", "description": "1-10, default 5"},
            },
            "required": ["query"],
        }}},
    {"type": "function", "function": {
        "name": "read_chunk",
        "description": ("Read the full text of a chunk by chunk_id. "
                       "Use after search_corpus to inspect a candidate's full content."),
        "parameters": {"type": "object",
            "properties": {"chunk_id": {"type": "string"}},
            "required": ["chunk_id"],
        }}},
]


def execute_tool(name: str, args: dict) -> dict:
    if name == "search_corpus":
        # The change vs Lab 07: pipe to search_corpus_v3.
        # rewrite_mode=None by default; the agent can request rewriting
        # via system-prompt instruction in a future iteration.
        return search_corpus_v3(query=args["query"], top_k=args.get("top_k", 5))
    if name == "read_chunk":
        return read_chunk(chunk_id=args["chunk_id"])
    return {"status": "error", "kind": "other", "detail": f"unknown tool: {name}"}


def _action_hash(name: str, args: dict) -> str:
    payload = name + "|" + json.dumps(args, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


SYSTEM_PROMPT = """You are a research assistant grounded in a specific document corpus.
Answer the user's question only from the corpus. Use search_corpus to find candidates,
then read_chunk to inspect their full text before answering. Phrase queries as 3-8
specific words. Refine if results are poor; do not repeat identical queries.
"""


def run_agent(question: str, max_steps: int = 8, verbose: bool = True) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    citations: list[dict] = []
    seen_actions: set[str] = set()

    for step in range(1, max_steps + 1):
        if verbose:
            print(f"\n── Step {step} ──")

        msg = chat_with_tools(messages, tools=TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg["content"]}
        if msg["tool_calls"]:
            entry["tool_calls"] = [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in msg["tool_calls"]
            ]
        messages.append(entry)

        if not msg["tool_calls"]:
            if verbose:
                print(f"  ◆ FINAL: {msg['content'][:140]}...")
            return {"answer": msg["content"], "citations": citations, "steps": step}

        for tc in msg["tool_calls"]:
            args = json.loads(tc["arguments"]) if tc["arguments"] else {}
            ah = _action_hash(tc["name"], args)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                              "detail": f"already called {tc['name']} with these args"}
                if verbose:
                    print(f"  ✗ {tc['name']}({args}) [REPEATED]")
            else:
                seen_actions.add(ah)
                tool_result = execute_tool(tc["name"], args)
                if verbose:
                    args_repr = str(args)[:60]
                    print(f"  → {tc['name']}({args_repr}) → {tool_result.get('status')}")
                if tc["name"] == "read_chunk" and tool_result.get("status") == "ok":
                    citations.append({
                        "chunk_id": tool_result["chunk_id"],
                        "doc_id": tool_result["doc_id"],
                        "title": tool_result["title"],
                    })
            messages.append({"role": "tool", "tool_call_id": tc["id"],
                            "content": json.dumps(tool_result)[:4000]})

    return {"answer": "Step cap reached.", "citations": citations, "steps": max_steps}


print("Agent ready with contextual retrieval + Lab 07 rerank pipeline.")


**Sample output:**

```
Agent ready with contextual retrieval + Lab 07 rerank pipeline.
```

Note that `execute_tool` calls `search_corpus_v3` with default `rewrite_mode=None`. Query rewriting adds latency; we keep it off-by-default and let the agent loop's natural query refinement do its work. For workloads where the first query *must* hit, you'd plumb `rewrite_mode="hyde"` through.

In [ ]:
# Run one of Lab 07's harder queries through the upgraded pipeline
query = ("What's the difference between using search as a tool versus using "
         "retrieval as a tool, and what failure modes does each have?")
print(f"QUERY: {query}")
print("=" * 70)
result = run_agent(query, verbose=True)
print("=" * 70)
print(f"\n✓ Steps: {result['steps']}, Citations: {len(result['citations'])}")
print(f"\nAnswer:\n{result['answer']}")
print("\nCitations:")
for c in result["citations"]:
    print(f"  - [{c['chunk_id']}] {c['title']}")


**Sample output:**

```
QUERY: What's the difference between using search as a tool versus using retrieval as a tool, and what failure modes does each have?
======================================================================

── Step 1 ──
  → search_corpus({'query': 'search tool retrieval tool difference failure modes'}) → ok

── Step 2 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:0'}) → ok

── Step 3 ──
  → read_chunk({'chunk_id': '04-search-vs-retrieval.md:2'}) → ok
  ◆ FINAL: Search and retrieval are two patterns that share an interface...
======================================================================

✓ Steps: 3, Citations: 2

Answer:
Search and retrieval are two patterns that share an interface (the agent
calls a function and gets back ranked text snippets) but differ structurally
on three axes...

Citations:
  - [04-search-vs-retrieval.md:0] Search and Retrieval: A Useful Distinction
  - [04-search-vs-retrieval.md:2] Search and Retrieval: A Useful Distinction
```

Three steps, two citations from the relevant document — same shape as Lab 07's result. The retrieval is slightly more confident under the hood (contextual indexes give the reranker better candidates), but the agent's behavior is identical.

This is the property we built toward: **the agent gets better retrieval without knowing anything about it**.

## Step 9 (stretch): Failure-mode walkthrough

The synthesis. Pick a few queries that are deliberately hard, walk
through the [retrieval-failure-modes](../../concepts/rag/retrieval-failure-modes.md)
decision tree on each, and demonstrate which intervention from
Lab 08 makes the difference.

In [ ]:
# Failure-mode test cases — deliberately constructed to trigger each FM
# Note: small corpus means we won't see all 8 FMs in stark form, but
# the diagnostic pattern is what to take away.

WALKTHROUGH_QUERIES = [
    {
        "query": "what does the embeddings document discuss about truncation",
        "expected_doc": "05-embeddings.md",
        "diagnosis": "FM 4 (paraphrase) — chunk talks about MiniLM's 256-wordpiece limit "
                     "but doesn't use the word 'truncation' prominently",
        "expected_fix": "contextual retrieval — chunk context names the doc topic",
    },
    {
        "query": "how does the system stop running the same tool repeatedly",
        "expected_doc": "01-agent-loop.md",
        "diagnosis": "FM 4 (paraphrase) — chunk uses 'repeated-action detector' "
                     "vocabulary",
        "expected_fix": "HyDE — hypothetical answer uses chunk's vocab",
    },
    {
        "query": "what is the difference between bi-encoder and cross-encoder retrieval and how does Lab 06 use it",
        "expected_doc": None,  # compound — sub-queries hit different docs
        "diagnosis": "FM 6 (compound query)",
        "expected_fix": "decomposition (or let the agent decompose)",
    },
]


def report_rank(name, results, expected_doc):
    if expected_doc is None:
        # For compound queries, just show what was retrieved
        return ", ".join({all_chunks[idx]["doc_id"] for idx, _ in results[:5]})
    for r, (idx, _) in enumerate(results, start=1):
        if all_chunks[idx]["doc_id"] == expected_doc:
            return f"rank {r}"
    return "miss"


for case in WALKTHROUGH_QUERIES:
    print("─" * 80)
    print(f"QUERY: {case['query']}")
    print(f"DIAGNOSIS: {case['diagnosis']}")
    print(f"PREDICTED FIX: {case['expected_fix']}")
    print()

    # Run each strategy, report the rank
    strategies = [
        ("Lab 07 baseline (hybrid + rerank)",
         lambda q: cross_encoder_rerank(
             q, hybrid_retrieve(emb_baseline, bm25_baseline, q, top_k=30), top_k=10)),
        ("+ contextual indexes",
         lambda q: cross_encoder_rerank(
             q, hybrid_retrieve(emb_contextual, bm25_contextual, q, top_k=30), top_k=10)),
        ("+ HyDE rewrite",
         lambda q: cross_encoder_rerank(
             q, hybrid_retrieve(emb_contextual, bm25_contextual, hyde_rewrite(q), top_k=30),
             top_k=10)),
        ("+ multi-query rewrite",
         lambda q: cross_encoder_rerank(q, (lambda rl: reciprocal_rank_fusion(rl, k=60)[:30])(
             {f"q{i}": hybrid_retrieve(emb_contextual, bm25_contextual, qq, top_k=30)
              for i, qq in enumerate(multi_query_rewrite(q))}), top_k=10)),
    ]

    for name, fn in strategies:
        results = fn(case["query"])
        report = report_rank(name, results, case["expected_doc"])
        print(f"  {name:<35} → {report}")
    print()


**Sample output:**

```
────────────────────────────────────────────────────────────────────────────────
QUERY: what does the embeddings document discuss about truncation
DIAGNOSIS: FM 4 (paraphrase) — chunk talks about MiniLM's 256-wordpiece limit but doesn't use the word 'truncation' prominently
PREDICTED FIX: contextual retrieval — chunk context names the doc topic

  Lab 07 baseline (hybrid + rerank)   → rank 1
  + contextual indexes                → rank 1
  + HyDE rewrite                      → rank 1
  + multi-query rewrite               → rank 1

────────────────────────────────────────────────────────────────────────────────
QUERY: how does the system stop running the same tool repeatedly
DIAGNOSIS: FM 4 (paraphrase) — chunk uses 'repeated-action detector' vocabulary
PREDICTED FIX: HyDE — hypothetical answer uses chunk's vocab

  Lab 07 baseline (hybrid + rerank)   → rank 1
  + contextual indexes                → rank 1
  + HyDE rewrite                      → rank 1
  + multi-query rewrite               → rank 1

────────────────────────────────────────────────────────────────────────────────
QUERY: what is the difference between bi-encoder and cross-encoder retrieval and how does Lab 06 use it
DIAGNOSIS: FM 6 (compound query)
PREDICTED FIX: decomposition (or let the agent decompose)

  Lab 07 baseline (hybrid + rerank)   → 05-embeddings.md, 04-search-vs-retrieval.md, 06-vector-indexes.md
  + contextual indexes                → 05-embeddings.md, 04-search-vs-retrieval.md, 06-vector-indexes.md
  + HyDE rewrite                      → 05-embeddings.md, 06-vector-indexes.md, 02-tool-design.md
  + multi-query rewrite               → 05-embeddings.md, 06-vector-indexes.md, 04-search-vs-retrieval.md
```

**Honest finding.** On this 55-chunk corpus, the reranker (from Lab 07) is so dominant that downstream interventions are mostly no-ops — the right chunk is already at rank 1. The mechanisms are *visible* (the rewrites change scores and candidate sets) but the *outcomes* are mostly unchanged because there's nowhere left to improve on this small, well-aligned dataset.

This is exactly what the lab brief warned about. The interventions don't disappear; they're just hiding under the reranker. Production corpora (10K-1M chunks) routinely have the failure modes this lab demonstrates, and the gains stack visibly.

The discipline to take away:

1. **Diagnose before intervening.** The decision tree in [retrieval-failure-modes.md](../../concepts/rag/retrieval-failure-modes.md) walks you through this.
2. **Each intervention targets a different failure mode.** Contextual retrieval helps chunks-out-of-context; HyDE helps vocabulary mismatch; decomposition helps compound queries.
3. **Stacking is sometimes free, sometimes redundant.** On small corpora with strong rerank, stacking helps less. On large messy corpora, it helps a lot. **Measure on your own corpus** before committing the LLM-call budget.

## ✓ Lab complete

You've now built every standard retrieval intervention covered in
mainstream RAG literature:

- **Lab 06:** bi-encoder + chunking + agent loop with citations.
- **Lab 07:** BM25 + RRF + MMR + cross-encoder rerank.
- **Lab 08:** contextual retrieval + HyDE + multi-query + decomposition.

The full pipeline — from corpus chunking to reranked top-k — covers
the ~95% of production RAG that doesn't require fine-tuning,
late-interaction models, or hosted reranker APIs.

Three properties worth internalizing before you move on:

1. **Each intervention targets a specific failure mode.** Don't stack
   them all by default. Read [retrieval-failure-modes.md](../../concepts/rag/retrieval-failure-modes.md);
   diagnose; then intervene.
2. **Index-time work is cheap on the second run.** Contextual retrieval
   pays its cost once and amortizes across every future query. Caching
   matters.
3. **Query rewriting is expensive on every query.** HyDE and multi-query
   each add LLM calls. They're worth it when retrieval is the
   bottleneck; they're net-negative when the LLM is the bottleneck or
   latency is the constraint.

### What to do next

- 🧠 **Take the quiz:** [`quizzes/agentic-rag/contextual-retrieval-and-query-rewriting.md`](../../quizzes/agentic-rag/contextual-retrieval-and-query-rewriting.md)
- 🧭 **Extend this lab.** Ideas:
  1. **Prompt-cached context generation.** Use Anthropic's prompt
     caching API to reduce the index-time LLM call cost by ~80%.
  2. **Custom contextualizer prompt** for your domain. The generic
     prompt works; domain-specific prompts often work better.
  3. **Conditional query rewriting.** Run baseline retrieval first;
     if top-1 score is below a threshold, run HyDE and retry.
  4. **Cheaper context model.** Use a smaller model for context
     generation than for answer generation. Same API, different model
     string.
- 🧭 **Continue Path 02 in future batches:** RAG evaluation primer,
  framework bridge, conversational RAG.
- 🧭 **Or move on:** Path 03 (Multi-Agent Systems) or Path 06
  (Evaluation & Observability). Both build on what you've learned.